In [1]:
import os
import re
from collections import Counter
from datetime import datetime, timedelta
from typing import Dict, List, Tuple

import pandas as pd


In [ ]:
DATASETS = ["Books", "Electronics", "Sports", "Tools"]
DATA_DIR = "./raw"
DATASET_NAME = "".join([d[0] for d in DATASETS])
OUTPUT_DIR = f"path/to/CGRec/{DATASET_NAME}"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Dataset mapping dictionary
dataset2fullname = {
    "Appliances": "Amazon_Appliances",
    "Arts": "Amazon_Arts_Crafts_and_Sewing",
    "Automotive": "Amazon_Automotive",
    "Beauty": "Amazon_All_Beauty",
    "Books": "Amazon_Books",
    "CDs": "Amazon_CDs_and_Vinyl",
    "Cell": "Amazon_Cell_Phones_and_Accessories",
    "Clothing": "Amazon_Clothing_Shoes_and_Jewelry",
    "Electronics": "Amazon_Electronics",
    "epinions": "epinions",
    "Fashion": "Amazon_AMAZON_FASHION",
    "food": "Food",
    "Food": "Amazon_Grocery_and_Gourmet_Food",
    "Garden": "Amazon_Patio_Lawn_and_Garden",
    "Gift": "Amazon_Gift_Cards",
    "Games": "Amazon_Video_Games",
    "Instruments": "Amazon_Musical_Instruments",
    "Kindle": "Amazon_Kindle_Store",
    "Kitchen": "Amazon_Home_and_Kitchen",
    "Luxury": "Amazon_Luxury_Beauty",
    "Magazine": "Amazon_Magazine_Subscriptions",
    "ml100k": "ml-100k",
    "Movies": "Amazon_Movies_and_TV",
    "Music": "Amazon_Digital_Music",
    "Office": "Amazon_Office_Products",
    "Pantry": "Amazon_Prime_Pantry",
    "Pet": "Amazon_Pet_Supplies",
    "Scientific": "Amazon_Industrial_and_Scientific",
    "Software": "Amazon_Software",
    "Sports": "Amazon_Sports_and_Outdoors",
    "Tools": "Amazon_Tools_and_Home_Improvement",
    "Toys": "Amazon_Toys_and_Games",
}


In [10]:
def _load_and_process_attrs(
    datasets: List[str], data_dir: str, dataset2fullname: Dict[str, str]
) -> Tuple[Dict[str, str], Dict[str, str]]:
    all_attr_dfs = []
    for idx, domain in enumerate(datasets):
        file_path = os.path.join(
            data_dir, dataset2fullname[domain], f"{dataset2fullname[domain]}.item"
        )
        if not os.path.exists(file_path):
            print(f"Warning: Attribute file not found for {domain} at {file_path}")
            continue

        df = pd.read_csv(
            file_path,
            sep="\t",
            header=0,
            usecols=[0, 1],
            names=["item", "category"],
            dtype={"item": str, "category": str},
        ).drop_duplicates()

        # Add domain suffix to item IDs
        suffix = chr(65 + idx)
        df["item"] = df["item"].astype(str) + f"_{suffix}"
        all_attr_dfs.append(df)

    if not all_attr_dfs:
        return {}, {}

    combined_df = pd.concat(all_attr_dfs, ignore_index=True)
    cats = combined_df["category"].str.split(",", expand=True)
    combined_df["cat1"] = cats[0]
    combined_df["cat2"] = cats[1]

    item_to_cat1 = dict(zip(combined_df["item"], combined_df["cat1"]))
    item_to_cat2 = dict(zip(combined_df["item"], combined_df["cat2"]))

    return item_to_cat1, item_to_cat2


# 1. Load and process item attributes
item_to_cat1, item_to_cat2 = _load_and_process_attrs(
    DATASETS, DATA_DIR, dataset2fullname
)


In [11]:
def _load_item_maps(dataset_name: str) -> Tuple[Dict[str, str], Dict[str, str]]:
    item_path = f"./{dataset_name}/{dataset_name}.item"
    if not os.path.exists(item_path):
        print(f"Error: Item file not found at {item_path}")
        return {}, {}

    with open(item_path, "r") as f:
        item_lines = [line.strip() for line in f.readlines()]
        item_id_map = {str(i + 1): item_lines[i] for i in range(len(item_lines))}
        item_domain_map = {
            str(i + 1): line.split("_")[-1] for (i, line) in enumerate(item_lines)
        }
    return item_id_map, item_domain_map

# 2. Load item ID mappings
item_id_map, item_domain_map = _load_item_maps(DATASET_NAME)

In [12]:
def _clean_text(text: str) -> str:
    if pd.isna(text) or text is None:
        return "xxx"
    
    text = str(text).replace('"', "").replace("\n", "").replace("\r", "")
    text = re.sub(r'\.+', '.', text)
    text = text + "." if not text.endswith(".") else text
    text = text.replace(" ", "_")
    return text if len(text) < 2000 else "xxx."


In [13]:
def _process_sequences(
    dataset_name: str,
    item_id_map: Dict[str, str],
    item_domain_map: Dict[str, str],
    item_to_cat1: Dict[str, str],
    item_to_cat2: Dict[str, str],
) -> Tuple[pd.DataFrame, Dict[str, Counter]]:
    train_path = f"./{dataset_name}/{dataset_name}.full.test.inter"
    if not os.path.exists(train_path):
        print(f"Error: Train file not found at {train_path}")
        return pd.DataFrame(), {}

    df = pd.read_csv(
        train_path,
        sep="\t",
        header=0,
        names=["user_id", "item_seq", "target_item"],
        dtype={"user_id": str, "item_seq": str, "target_item": str},
    )

    # Create a list of all item IDs
    df["all_items"] = df.apply(
        lambda row: row["item_seq"].split() + [row["target_item"]], axis=1
    )

    # Explode the list and map the attributes
    df_expanded = df.explode("all_items").reset_index()
    df_expanded.rename(
        columns={"all_items": "item_id", "index": "original_index"}, inplace=True
    )

    df_expanded["raw_item"] = df_expanded["item_id"].map(item_id_map)
    df_expanded["type"] = df_expanded["item_id"].map(item_domain_map)

    # Use a vectorized approach with .map and a cleaning function
    df_expanded["cat1"] = df_expanded["raw_item"].map(item_to_cat1).apply(_clean_text)
    df_expanded["cat2"] = df_expanded["raw_item"].map(item_to_cat2).apply(_clean_text)

    # Create timestamps based on sequence order
    base_time = datetime(2025, 1, 1, 0, 0, 0)
    time_interval = timedelta(hours=1)
    df_expanded["unix_time"] = (
        df_expanded.groupby("original_index")
        .cumcount()
        .apply(lambda i: int((base_time + i * time_interval).timestamp()))
    )

    # Aggregate back into sequences
    amazon_seq_df = (
        df_expanded.groupby("user_id", sort=False)
        .agg(
            item=("raw_item", lambda x: ",".join(x)),
            type=("type", lambda x: ",".join(x)),
            cat1=("cat1", lambda x: ",".join(x)),
            cat2=("cat2", lambda x: ",".join(x)),
            unix_time=("unix_time", lambda x: ",".join(map(str, x))),
            list_len=("raw_item", "count"),
        )
        .reset_index()
    )

    # Count occurrences for vocabularies
    item_counter = Counter(df_expanded["raw_item"])
    cat1_counter = Counter(df_expanded["cat1"])
    cat2_counter = Counter(df_expanded["cat2"])
    # type_counter = Counter(df_expanded["type"])

    unique_items_df = df_expanded[["raw_item", "type"]].drop_duplicates()
    type_counter = Counter(unique_items_df["type"])

    return amazon_seq_df, {
        "item": item_counter,
        "type": type_counter,
        "cat1": cat1_counter,
        "cat2": cat2_counter,
    }


# 3. Process user sequences and generate vocabularies
amazon_seq_df, vocabs = _process_sequences(
    DATASET_NAME, item_id_map, item_domain_map, item_to_cat1, item_to_cat2
)


In [14]:
def _save_dataframes(
    amazon_seq_df: pd.DataFrame,
    vocabs: Dict[str, Counter],
    output_dir: str,
    dataset_name: str,
):
    # Save sequence data
    amazon_seq_path = os.path.join(output_dir, f"{dataset_name}_seq.txt")
    amazon_seq_df.to_csv(amazon_seq_path, sep="\t", index=True)
    print(f"Saved sequences  to {amazon_seq_path}")

    # Helper function to save vocabulary DataFrames
    def save_vocab(name: str, counter: Counter, sort_key=None):
        df = pd.DataFrame({name: list(counter.keys()), "cnt": list(counter.values())})
        if sort_key:
            df = df.sort_values(name, key=sort_key, ignore_index=True)
        else:
            df = df.sort_values(name, ignore_index=True)
        path = os.path.join(output_dir, f"voca_{name}.txt")
        df.to_csv(path, sep="\t", index=True)
        print(f"Saved vocabulary to {path}")

    save_vocab("item", vocabs["item"], sort_key=lambda x: x.str.split("_").str[-1])
    save_vocab("type", vocabs["type"])
    save_vocab("cat1", vocabs["cat1"])
    save_vocab("cat2", vocabs["cat2"])


# 4. Save the results
_save_dataframes(amazon_seq_df, vocabs, OUTPUT_DIR, DATASET_NAME)
print("\nData processing complete.")


Saved sequences  to ./CGRec/BEST/BEST_seq.txt
Saved vocabulary to ./CGRec/BEST/voca_item.txt
Saved vocabulary to ./CGRec/BEST/voca_type.txt
Saved vocabulary to ./CGRec/BEST/voca_cat1.txt
Saved vocabulary to ./CGRec/BEST/voca_cat2.txt

Data processing complete.
